## Tools in LangChain

A **tool** is a function that a model can call to do something it can't do on its
own — like search the web, run code, query a database, or call an API. Tools are
what turn a plain chat model into an **agent**.

### Why tools?

An LLM only knows what it was trained on and can only produce text. Tools let it
**take actions** and fetch **live/real data**.

| Without tools | With tools |
|---------------|-----------|
| Answers from memory only | Can fetch live data (weather, DB, APIs) |
| Text in, text out | Can run code, do math, call services |
| No real-world actions | Can act on the world |


### 1. Defining a tool with `@tool`

The easiest way is the `@tool` decorator. The **docstring** tells the model when
and how to use it.

```python
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers and return the result."""
    return a + b

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    return f"The weather in {city} is sunny, 25°C."
```

Each tool has:
- a **name** (from the function name)
- a **description** (from the docstring)
- an **input schema** (from the type hints)

```python
print(add.name)         # add
print(add.description)  # Add two numbers and return the result.
print(add.args)         # {'a': {...}, 'b': {...}}
```

### 2. Calling a tool directly

```python
result = add.invoke({"a": 3, "b": 5})
print(result)   # 8
```

### 3. Giving tools to a model (tool calling)

The model decides **which** tool to call and **what arguments** to pass. It does
not run the tool itself — it returns a request, and you execute it.

```python
tools = [add, get_weather]
model_with_tools = model.bind_tools(tools)

response = model_with_tools.invoke("What is 12 plus 30?")
print(response.tool_calls)
# [{'name': 'add', 'args': {'a': 12, 'b': 30}, 'id': '...'}]
```

### 4. Using tools inside an agent

An agent handles the full loop automatically: **model → pick tool → run tool →
feed result back → final answer.**

```python
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[add, get_weather],
    system_prompt="You are a helpful assistant.",
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "What's the weather in Bengaluru?"}]
})
print(result["messages"][-1].content)
```

### How the agent tool loop works

```text
User question
     │
     ▼
Model decides: "I need a tool"
     │
     ▼
Tool is called with arguments
     │
     ▼
Tool result returned to the model
     │
     ▼
Model writes the final answer
```

### Types of tools you can use

| Type | Example |
|------|---------|
| Custom functions | Your own `@tool` functions |
| Built-in tools | Web search, Wikipedia, Python REPL |
| API wrappers | Weather, maps, company APIs |
| Retrievers | Search your own documents (RAG) |

### Key notes

- The **docstring matters** — it's how the model knows what the tool does. Write
  it clearly.
- **Type hints** define the input schema, so the model passes correct arguments.
- `bind_tools()` = model *decides* the call; an **agent** = full loop that also
  *runs* the tool and continues.
- Keep tools **small and single-purpose** — easier for the model to use correctly.

In [2]:
import os, httpx
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

http_client = httpx.Client(verify=False)          # ⚠️ dev only

model = ChatOpenAI(                                # 👈 REBUILD model here
    model="llama3.2",
    base_url=os.getenv("GE_BASE_URL"),
    api_key=os.getenv("GE_API_KEY"),
    temperature=0,
    http_client=http_client,                       # 👈 attach the client
)
model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.2', 'langchain': '1.3.14', 'langchain-openai': '1.4.1'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x00000271A1823830>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000271A1825B20>, root_client=<openai.OpenAI object at 0x00000271A138A870>, root_async_client=<openai.AsyncOpenAI object at 0x00000271A135B8F0>, model_name='llama3.2', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='http://genai-imaging-lab.apps.ge-healthcare.net:4000/v1', openai_proxy=None, http_client=<httpx.Client object at 0x00000271A1684EF0>, stream_chunk_timeout=120.0)

In [6]:
from langchain.tools import tool


@tool
def subtract_numbers(num1: float, num2: float) -> float:
    """Subtracts two numbers."""
    return num1 - num2

@tool
def multiply_numbers(num1: float, num2: float) -> float:
    """Multiplies two numbers."""
    return num1 * num2

@tool
def get_current_weather(location: str) -> str:
    """Fetches the current weather for a given location."""
    # Here you would implement the logic to fetch weather data.
    # For demonstration purposes, we'll return a mock response.
    return f"The current weather in {location} is sunny with a temperature of 25°C."



In [32]:
tools = [ subtract_numbers, multiply_numbers, get_current_weather]
tools_by_name = {t.name: t for t in tools}
model_with_tools = model.bind_tools(tools)

In [38]:
res1 = model_with_tools.invoke("What is 2 minus 8?")
res1

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 243, 'total_tokens': 268, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'ollama_chat/llama3.2', 'system_fingerprint': None, 'id': 'chatcmpl-3ba1194c-9013-4300-9144-4a1ff220bc17', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fb45e-5c0c-75a0-b7c6-8014683e6d41-0', tool_calls=[{'name': 'subtract_numbers', 'args': {'num1': '2', 'num2': '8'}, 'id': 'call_vr37d77s', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 243, 'output_tokens': 25, 'total_tokens': 268, 'input_token_details': {}, 'output_token_details': {}})

In [ ]:
for tool_call in res1.tool_calls:
    print(f"Tool called: {tool_call['name']} with arguments: {tool_call['args']}")


Tool called: subtract_numbers with arguments: {'num1': '2', 'num2': '8'}


In [43]:
res2 = model_with_tools.invoke("What is weather in New York?")
res2

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 242, 'total_tokens': 261, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'ollama_chat/llama3.2', 'system_fingerprint': None, 'id': 'chatcmpl-d384f8a6-a934-4226-835d-8a1d61cd9da5', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fb460-45cf-70c3-84bd-b32fc18ae73f-0', tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'New York'}, 'id': 'call_v8zk94lh', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 242, 'output_tokens': 19, 'total_tokens': 261, 'input_token_details': {}, 'output_token_details': {}})

In [42]:
for tool_call in res1.tool_calls:
    print(f"Tool called: {tool_call['name']} with arguments: {tool_call['args']}")

Tool called: get_current_weather with arguments: {'location': 'New York'}


## Tool Execution Loop

#### Step 1: Model generates the Tool calls
- You start a conversation list with the user's question.
- The model responds — but instead of text, it returns a tool call request (subtract_numbers with num1=2, num2=8).
- You append that AI message to messages so the conversation history grows.

**At this point ai_msg.content is empty — the request lives in ai_msg.tool_calls.**

In [46]:

messages = [{"role": "user", "content": "What is 2 minus 8?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#### Step 2 : Execute the Tool and generate the result 

- Loop over the tool call(s) the model requested.
- subtract_numbers.invoke(tool_call) actually runs your function → computes 2 - 8 = -6.
- Passing the whole tool_call (not just the args) makes .invoke() return a proper ToolMessage (with the - matching tool_call_id), which you append to messages.

**Now the conversation contains: user question → tool request → tool result.**

In [47]:

for tool_call in ai_msg.tool_calls:
    # Execute the tool with generated arguments
    tool_result = subtract_numbers.invoke(tool_call)
    messages.append(tool_result)


#### Step 3 : pass the result back to the model to generate the final response

- The model now sees the whole history, including the tool's result (-6).
- It turns that raw number into a human sentence:

In [48]:
final_response = model_with_tools.invoke(messages)
final_response.content

'The result of 2 minus 8 is -6.'

In [49]:
messages

[{'role': 'user', 'content': 'What is 2 minus 8?'},
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 243, 'total_tokens': 268, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'ollama_chat/llama3.2', 'system_fingerprint': None, 'id': 'chatcmpl-52c1afc6-7339-489e-9019-672e1d695fde', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fb46c-e52b-7bc3-8bb3-14acf137a1e3-0', tool_calls=[{'name': 'subtract_numbers', 'args': {'num1': '2', 'num2': '8'}, 'id': 'call_n4m0d4m9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 243, 'output_tokens': 25, 'total_tokens': 268, 'input_token_details': {}, 'output_token_details': {}}),
 ToolMessage(content='-6.0', name='subtract_numbers', tool_call_id='call_n4m0d4m9')]